# 04 – Monthly probability fields (used by gap-filling step 4) – run once

For each **calendar month**, all the binary products of this month over 1984–2021 are used:
- **clear** = number of years in which the pixel is 0 or 1 (not cloud);
- **clear < 29 years** → NA (not enough observations);
- **snow in 100 % of the clear years** → **1**;
- **no snow in 100 % of the clear years** → **2**;
- **otherwise** → NA.

The threshold (100 %) is set by `frac`.

**Input:** the original binary stacks 1984–2021 `raster_stack_predi_<season>_<part>.tif` (Zenodo; 1 snow, 0 no snow, NA cloud). The date of each band is given in `data/original_binary_stacks_dates.csv`.
- DJF is split in time: `_a` = bands 1–73, `_b` = bands 73–105 (band 73 of `_a` duplicates band 1 of `_b`).
- MAM, JJA and SON are split in western and eastern halves.

**Outputs:**
- `data/aux/probability_field_<month>.tif` on the NDSI grid (1, 2, 0 = NA), used by `05_gapfilling_steps2-7.ipynb`;
- `outputs/probability_fields_netcdf/<Month>.nc`, `<Month>_a.nc` (west) and `<Month>_b.nc` (east): NetCDF files (variable `layer`, grid of the original products 14234 × 6200), as written by the original R script.

In [ ]:
import os, time
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin
import matplotlib.pyplot as plt

stacks_dir = '../data/original_binary_stacks'          # from Zenodo
dates_csv = '../data/original_binary_stacks_dates.csv'  # date of each band of the stacks
out_pattern = '../data/aux/probability_field_%02d.tif'
min_clear = 29          # minimum number of clear years
frac = 1.0              # 1.0 = always snow / always no snow

W_R, H_R = 14234, 6200
half = W_R // 2
dates = pd.read_csv(dates_csv)
dates = dates[dates.note.isna()]                       # band 73 of DJF_a duplicates band 1 of DJF_b
print(dates.groupby('season').size())

## Number of clear years with and without snow, per calendar month

In [ ]:
n1 = {m: np.zeros((H_R, W_R), 'uint16') for m in range(1, 13)}     # years with snow
n0 = {m: np.zeros((H_R, W_R), 'uint16') for m in range(1, 13)}     # years without snow
used = {m: [] for m in range(1, 13)}

t0 = time.time()
for f, d in dates.groupby('file', sort=False):
    col0 = half if d.part.iloc[0] == 'east half' else 0
    with rasterio.open(os.path.join(stacks_dir, f)) as r:
        for _, row in d.iterrows():
            a = r.read(int(row.band))
            cols = slice(col0, col0 + a.shape[1])
            n1[row.month][:, cols] += (a == 1)
            n0[row.month][:, cols] += (a == 0)
            if col0 == 0:
                used[row.month].append(row.year)
    print('%s done (%.0f s)' % (f, time.time() - t0))

for m in range(1, 13):
    print('month %02d: %i products %s ... %s' % (m, len(used[m]), used[m][:3], used[m][-3:]))

## Fields 1 / 2 on the NDSI grid

In [ ]:
T_R = rasterio.open(os.path.join(stacks_dir, 'raster_stack_predi_DJF_a.tif')).transform
T_N = from_origin(5.699957268017272, 47.85026882419969, 0.0003407435000010325, 0.00034046861818428666)
W_N, H_N = 14234, 6199
MASK = rasterio.open('../data/aux/mask_switzerland_without_lakes.tif').read(1) == 1
n_ch = MASK.sum()

# also written as NetCDF, as in the original R script: <Month>.nc and the halves <Month>_a.nc (west) / <Month>_b.nc (east),
# variable 'layer' (1 = snow, 2 = no snow, NaN), on the grid of the R products (14234 x 6200)
import xarray as xr
r_dir = '../outputs/probability_fields_netcdf'
os.makedirs(r_dir, exist_ok=True)
month_names = {1: 'January', 2: 'February', 3: 'March', 4: 'April', 5: 'May', 6: 'June', 7: 'July',
               8: 'August', 9: 'September', 10: 'October', 11: 'November', 12: 'December'}
lon_R = T_R.c + (np.arange(W_R) + 0.5) * T_R.a
lat_R = T_R.f + (np.arange(H_R) + 0.5) * T_R.e

def write_R_names(field, m):
    layer = np.where(field == 0, np.nan, field).astype('float32')
    for suffix, cols in [('', slice(0, W_R)), ('_a', slice(0, half)), ('_b', slice(half, W_R))]:
        ds = xr.Dataset({'layer': (('latitude', 'longitude'), layer[:, cols])},
                        coords={'latitude': lat_R, 'longitude': lon_R[cols]})
        ds['layer'].attrs['description'] = '1 = snow in 100 % of the clear years, 2 = no snow in 100 % of the clear years (>= 29 clear years)'
        ds.attrs['crs'] = 'EPSG:4326'
        ds.to_netcdf('%s/%s%s.nc' % (r_dir, month_names[m], suffix), encoding={'layer': {'zlib': True}})

pct = {}
for m in range(1, 13):
    clear = n0[m] + n1[m]
    ok = clear >= min_clear
    field = np.zeros((H_R, W_R), 'uint8')
    field[ok & (n1[m] >= frac * clear)] = 1
    field[ok & (n0[m] >= frac * clear)] = 2
    write_R_names(field, m)
    out = np.zeros((H_N, W_N), 'uint8')
    reproject(field, out, src_transform=T_R, src_crs='EPSG:4326', dst_transform=T_N, dst_crs='EPSG:4326',
              src_nodata=0, dst_nodata=0, resampling=Resampling.nearest)
    out[~MASK] = 0
    with rasterio.open(out_pattern % m, 'w', driver='GTiff', width=W_N, height=H_N, count=1, dtype='uint8',
                       crs='EPSG:4326', transform=T_N, nodata=0, compress='deflate') as dst:
        dst.write(out, 1)
    pct[m] = (100 * (out == 1).sum() / n_ch, 100 * (out == 2).sum() / n_ch)
    print('month %02d: snow (1) %.3f %% | no snow (2) %.3f %%' % (m, *pct[m]))

## Check: comparison with the percentages of the original R script

In [ ]:
R_land = [1.827911e-04, 5.785092e-03, 3.804772e-02, 1.089507e+00, 2.723145e+00, 1.831320e+01, 3.186041e+01, 4.370062e+01, 4.993906e+01, 3.172566e+01, 6.598915e+00, 3.548840e-01]
R_snow = [0.2680903, 0.8121286, 10.0396143, 7.4514205, 2.8339613, 0.6216158, 0.3365012, 0.2492085, 0.2236622, 0.2211797, 0.3184542, 0.1151757]
order = [12, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]                  # Dec ... Nov as in the R script
pd.DataFrame({'month': order,
              'snow R (%)': R_snow, 'snow Python (%)': [round(pct[m][0], 4) for m in order],
              'no snow R (%)': R_land, 'no snow Python (%)': [round(pct[m][1], 4) for m in order]})

In [ ]:
fig, ax = plt.subplots(3, 4, figsize=(24, 12))
for a, m in zip(ax.flat, order):
    f = rasterio.open(out_pattern % m).read(1)[::6, ::6].astype(float); f[f == 0] = np.nan
    a.imshow(f, cmap=plt.matplotlib.colors.ListedColormap(['#3a7bd5', '#9bc36b']), vmin=1, vmax=2, interpolation='nearest')
    a.set_title('month %02d' % m); a.axis('off')
plt.tight_layout()